# 🧑‍🍳 Neosantara AI Cookbook: Stateful Conversations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neosantara-xyz/examples/blob/main/cookbook/advanced/stateful-conversations.ipynb)

This notebook demonstrates how to manage **Stateful Conversations** using the Neosantara Responses API (`v1/responses`). 

Unlike traditional Chat Completions where you must send the entire message history with every request, the Responses API allows you to maintain conversation state on the server. This results in:
- **Lower Latency & Cost**: No need to re-upload history tokens.
- **Simpler Client Logic**: The server handles the "messages" array for you.
- **Context Continuity**: The model remembers previous turns, tools, and reasoning automatically.

### 📖 Documentation
- [Conversation Management Guide](https://docs.neosantara.xyz/en/conversation-management)
- [Responses API Reference](https://docs.neosantara.xyz/api-reference/responses/generate-a-response)

In [ ]:
# Install the OpenAI SDK (v1.50.0+ required for Responses API)
!pip install "openai>=1.50.0"

# Setup API Key (for Google Colab)
try:
    from google.colab import userdata
    import os
    os.environ["NEOSANTARA_API_KEY"] = userdata.get('NEOSANTARA_API_KEY')
    print("API Key loaded from Google Colab userdata.")
except ImportError:
    import os
    if "NEOSANTARA_API_KEY" in os.environ:
        print("API Key loaded from environment variables.")
    else:
        print("Not running in Google Colab and NEOSANTARA_API_KEY not found in environment.")

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("NEOSANTARA_API_KEY"),
    base_url="https://api.neosantara.xyz/v1"
)

### 1️⃣ Turn 1: Starting the Conversation

To start a stateful conversation, we set `store=True` (this is the default behavior). The API will return a `response.id` that we can use to reference this state in the next turn.

In [ ]:
# We introduce ourselves and ask the system to store this context
response1 = client.responses.create(
    model="nusantara-base",
    input="My name is Alice. I am a software engineer studying Indonesian culture.",
    store=True
)

print(f"Response 1 ID: {response1.id}")
print("---")
print(response1.output_text)

### 2️⃣ Turn 2: Continuing with `previous_response_id`

In the next turn, we simply pass the `id` from the previous response. We **do not** need to resend the information that our name is Alice or that we are a software engineer.

In [ ]:
# We ask a follow-up question referencing the previous response ID
response2 = client.responses.create(
    model="nusantara-base",
    input="What is a traditional Indonesian concept that relates to collaboration in my field?",
    previous_response_id=response1.id
)

print(f"Response 2 ID: {response2.id}")
print("---")
print(response2.output_text)

### 🧠 Context Management Benefits

As you can see in the output above, the model successfully linked the second question to the first turn's context. 

**Key Takeaways:**
1. **Efficiency**: Only the new tokens are billed for input (plus server-side context retrieval which is typically cheaper than re-sending full history).
2. **Branching**: You can branch conversations by referencing the same `previous_response_id` multiple times to explore different paths.
3. **Stateless Option**: If you have privacy requirements (Zero Data Retention), you can set `store=False` to ensure no state is kept on Neosantara's servers.